<a href="https://colab.research.google.com/github/quantatrisk/quantatrisk.github.io/blob/main/docling_conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# 1. Install System Dependencies and Docling
!pip install -q docling pillow
!apt-get install -y -q tesseract-ocr libtesseract-dev

Reading package lists...
Building dependency tree...
Reading state information...
libtesseract-dev is already the newest version (4.1.1-2.1build1).
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [16]:
import os
import re
import urllib.parse
from pathlib import Path
from google.colab import files
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling_core.types.doc import PictureItem, TableItem

# 1. Upload PDF files
uploaded = files.upload()
if not uploaded:
    print("No files uploaded.")
else:
    # 2. Configure Extraction Pipeline
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = True
    pipeline_options.do_table_structure = True
    pipeline_options.do_formula_enrichment = True
    pipeline_options.generate_page_images = True
    pipeline_options.generate_picture_images = True

    converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
    )

    # 3. Process each file
    for filename in uploaded.keys():
        print(f"\n--- Processing: {filename} ---")
        doc_slug = Path(filename).stem
        image_folder = f"images_{doc_slug}"
        Path(image_folder).mkdir(exist_ok=True)

        result = converter.convert(filename)

        # Save Images
        image_count = 0
        for element, _level in result.document.iterate_items():
            if isinstance(element, (PictureItem, TableItem)):
                image_count += 1
                img = element.get_image(result.document)
                if img:
                    img.save(Path(image_folder) / f"{doc_slug}_fig_{image_count}.png", "PNG")

        # Export Markdown with encoded image paths (PyCharm compatible)
        raw_md = result.document.export_to_markdown(image_placeholder="DOCLING_IMG_PLACEHOLDER")

        state = {'count': 0}
        def replace_with_encoded_path(match):
            state['count'] += 1
            path = f"{image_folder}/{doc_slug}_fig_{state['count']}.png"
            encoded_path = urllib.parse.quote(path)
            return f"![Figure]({encoded_path})"

        final_md = re.sub(r"DOCLING_IMG_PLACEHOLDER", replace_with_encoded_path, raw_md)

        with open(f"{doc_slug}_final.md", "w", encoding="utf-8") as f:
            f.write(final_md)

    print("\n✅ Extraction and path encoding complete!")

[INFO] 2026-03-17 08:46:17,305 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-17 08:46:17,306 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-03-17 08:46:17,347 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-17 08:46:17,347 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth


Saving Chapter 9 - Oil as a World Market.pdf to Chapter 9 - Oil as a World Market (1).pdf

--- Processing: Chapter 9 - Oil as a World Market (1).pdf ---


[INFO] 2026-03-17 08:46:17,585 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-17 08:46:17,587 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-03-17 08:46:17,592 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-17 08:46:17,592 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-17 08:46:17,676 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-17 08:46:17,677 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-03-17 08:46:17,753 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.pth
[INFO] 2026-03-17 08:46:17,754 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.pth



✅ Extraction and path encoding complete!


In [17]:
# 4. Zip and Download everything
!zip -r final_extracted_package.zip *_final.md images_*/

from google.colab import files
if os.path.exists('final_extracted_package.zip'):
    files.download('final_extracted_package.zip')
else:
    print("No files found to download.")

  adding: Chapter 9 - Oil as a World Market (1)_final.md (deflated 64%)
  adding: images_Chapter 10 - The Gas Market as the Energy Market/ (stored 0%)
  adding: images_Chapter 10 - The Gas Market as the Energy Market/Chapter 10 - The Gas Market as the Energy Market_fig_9.png (deflated 6%)
  adding: images_Chapter 10 - The Gas Market as the Energy Market/Chapter 10 - The Gas Market as the Energy Market_fig_1.png (deflated 1%)
  adding: images_Chapter 10 - The Gas Market as the Energy Market/Chapter 10 - The Gas Market as the Energy Market_fig_10.png (deflated 2%)
  adding: images_Chapter 10 - The Gas Market as the Energy Market/Chapter 10 - The Gas Market as the Energy Market_fig_5.png (deflated 6%)
  adding: images_Chapter 10 - The Gas Market as the Energy Market/Chapter 10 - The Gas Market as the Energy Market_fig_4.png (deflated 5%)
  adding: images_Chapter 10 - The Gas Market as the Energy Market/Chapter 10 - The Gas Market as the Energy Market_fig_3.png (deflated 9%)
  adding: imag

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import re
from pathlib import Path

for filename in uploaded.keys():
    print(f"\n--- Inserting Image Links: {filename} ---")
    doc_slug = Path(filename).stem
    image_folder = f"images_{doc_slug}"

    # 1. Convert to get the document
    result = converter.convert(filename)

    # 2. Export with a unique text placeholder
    raw_md = result.document.export_to_markdown(image_placeholder="DOCLING_IMG_PLACEHOLDER")

    # 3. Post-process to insert incrementing image paths
    # Using a list to hold the counter avoids 'nonlocal' scoping issues
    state = {'count': 0}

    def replace_placeholder(match):
        state['count'] += 1
        path = f"{image_folder}/{doc_slug}_fig_{state['count']}.png"
        return f"![Figure]({path})"

    # Regex find and replace
    final_md = re.sub(r"DOCLING_IMG_PLACEHOLDER", replace_placeholder, raw_md)

    output_path = f"{doc_slug}_with_images.md"
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(final_md)

print("\n✅ Success! Markdown files with embedded image links are ready.")


--- Inserting Image Links: Chapter 9 - Oil as a World Market.pdf ---

--- Inserting Image Links: Chapter 10 - The Gas Market as the Energy Market.pdf ---


[WARNING] 2026-03-17 08:22:12,567 [RapidOCR] main.py:125: The text detection result is empty



--- Inserting Image Links: Oil Logistics.pdf ---

✅ Success! Markdown files with embedded image links are ready.


In [14]:
import urllib.parse
import re
from pathlib import Path

# This script updates the Markdown files to use URL-encoded paths (e.g., ' ' -> '%20')
# which is more compatible with IDE renderers like PyCharm.

md_files = list(Path('.').glob('*_with_images.md'))

for md_file in md_files:
    print(f"Encoding paths in: {md_file.name}")
    with open(md_file, 'r', encoding='utf-8') as f:
        content = f.read()

    # Find Markdown image syntax: ![...] (path)
    def encode_path(match):
        alt_text = match.group(1)
        path = match.group(2)
        # URL encode the path (keeps slashes but fixes spaces)
        encoded_path = urllib.parse.quote(path)
        return f'![{alt_text}]({encoded_path})'

    # Regex to find ![alt](path)
    new_content = re.sub(r'\!\[(.*?)\]\((.*?)\)', encode_path, content)

    output_name = md_file.stem + "_encoded.md"
    with open(output_name, 'w', encoding='utf-8') as f:
        f.write(new_content)

print("\n✅ Done! Download the '_encoded.md' files to try in PyCharm.")

Encoding paths in: Oil Logistics_with_images.md
Encoding paths in: Chapter 9 - Oil as a World Market_with_images.md
Encoding paths in: Chapter 10 - The Gas Market as the Energy Market_with_images.md

✅ Done! Download the '_encoded.md' files to try in PyCharm.
